
# <p style="text-align: center;">Predict Drought Decrees</p>

## Import libraries

In [106]:
#!pip install imblearn

In [82]:
import pandas as pd
import numpy as np

import glob
import os

import requests

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

import zipfile

import xarray as xr

import folium
from folium.plugins import MarkerCluster

import pandas as pd
from geopy.geocoders import Nominatim

import geopandas as gpd
import cartopy.crs as ccrs

import imageio

from sklearn.neighbors import KDTree
from sklearn.neighbors import BallTree

from sklearn.preprocessing import RobustScaler

## Set parameters

In [83]:
decree_filename_base = 'arrete_'
decrees_folder_name = './../../data/raw/decrees'
communes_folder_name = './../../data/raw/opendatasoft'
weather_folder_name = './../../data/raw/weather/era5'
weather_ncfiles_folder_name = './../../data/raw/weather/era5/ncfiles2'
processed_data_folder_name = './../../data/processed'
output_data_folder_name = './../../data/processed/output'
decrees_filename = 'decrees.parquet'
decrees_locations_filename = 'decrees_locations.parquet'
communes_csv_filename = 'correspondance-code-insee-code-postal-202410.csv'
weather_filename = 'weather2.parquet'
weather_shema_filename = 'weather_shema2.csv'
weather_yearly_filename = 'weather2_yearly.parquet'
weather_yearly_shema_filename = 'weather_shema2_yearly.csv'
drought_filename = 'drought.parquet'
drought_shema_filename = 'drought_shema.csv'
drought_yearly_filename = 'drought_yearly.parquet'
drought_yearly_shema_filename = 'drought_yearly_shema.csv'
drought_commune_clean_filename = 'drought_commune_clean.parquet'
drought_commune_clean_shema_filename = 'drought_commune_clean_shema.csv'
drought_weather_filename = 'drought_weather.parquet'
drought_weather_shema_filename = 'drought_weather_shema.csv'
#weather_zip_file = "193fcd51a8958175843ecbbcaba057c8.zip"

## Load data

| Column    | Description                                      |
|-----------|--------------------------------------------------|
| date      | The date of the observation.                     |
| latitude  | The latitude coordinate of the observation point.|
| longitude | The longitude coordinate of the observation point.|
| number    | A unique identifier for the observation.         |
| expver    | Experiment version number.                       |
| u10       | 10-meter U-component of wind (eastward wind).    |
| v10       | 10-meter V-component of wind (northward wind).   |
| t2m       | 2-meter temperature (air temperature at 2 meters above the surface).|
| sp        | Surface pressure.                                |
| tp        | Total precipitation.                             |
| e         | Evaporation.                                     |
| sro       | Surface runoff.                                  |
| tcrw      | Total column water vapor.                        |
| stl1      | Soil temperature level 1.                        |
| stl2      | Soil temperature level 2.                        |
| slt       | Soil type.                                       |
| swvl1     | Volumetric soil water layer 1.                   |
| swvl2     | Volumetric soil water layer 2.                   |
| cvh       | High vegetation cover.                           |
| cvl       | Low vegetation cover.                            |
| tvh       | High vegetation type.                            |
| tvl       | Low vegetation type.                             |


## Import Data

In [84]:
# read the dataframe from parquet
import pandas as pd

df = pd.read_parquet(os.path.join(processed_data_folder_name, drought_weather_filename))




In [85]:
os.path.join(processed_data_folder_name, drought_weather_shema_filename)

'./../../data/processed/drought_weather_shema.csv'

In [86]:
# Load the schema (data types) from the file
schema = pd.read_csv(os.path.join(processed_data_folder_name, drought_weather_shema_filename), index_col=0).squeeze("columns")

In [87]:
# Apply the schema to the loaded dataframe
df = df.astype(schema.to_dict())

In [101]:
# Remove rows where applied_1 is equal to 0
df = df[df['applied_1'] != 0]

print(df)


      insee_1           nom_commune_1  latitude_1  longitude_1  year_1  \
9       10010            ARREMBECOURT   48.543296     4.606490    2000   
10      10011             ARRENTIERES   48.267992     4.739104    2000   
16      10018                   AUXON   48.092318     3.923462    2000   
23      10024                 AVREUIL   48.053139     3.998028    2000   
25      10026         BAILLY-LE-FRANC   48.522491     4.659134    2000   
...       ...                     ...         ...          ...     ...   
35990   95592             SERAINCOURT   49.042140     1.878670    2023   
35992   95598  SOISY-SOUS-MONTMORENCY   48.988408     2.300500    2023   
35994   95607                 TAVERNY   49.026732     2.221161    2023   
36002   95637                 VAUREAL   49.029507     2.024466    2023   
36011   95678           VILLIERS-ADAM   49.070289     2.239509    2023   

       applied_1  decision_1 Code Postal_1   Département_1  \
9              1           0         10330       

In [88]:
df.head()

,insee_1,nom_commune_1,latitude_1,longitude_1,year_1,applied_1,decision_1,Code Postal_1,Département_1,Région_1,...,stl2_mean_2,slt_sum_2,slt_mean_2,swvl1_sum_2,swvl1_mean_2,swvl2_sum_2,swvl2_mean_2,latitude_rad_2,longitude_rad_2,distance_km
0,10002,AILLEVILLE,48.255000,4.694493,2000,0,0,10200,['AUBE'],['CHAMPAGNE-ARDENNE'],...,11.360728,36.0,3.0,4.290780,0.357565,4.225537,0.352128,0.842121,0.082903,4.147094
1,10003,AIX-EN-OTHE,48.197575,3.736328,2000,0,0,10160,['AUBE'],['CHAMPAGNE-ARDENNE'],...,11.673390,24.0,2.0,4.221337,0.351778,4.156231,0.346353,0.842121,0.065450,5.916774
2,10004,ALLIBAUDIERES,48.586400,4.122928,2000,0,0,10700,['AUBE'],['CHAMPAGNE-ARDENNE'],...,11.781789,24.0,2.0,4.192894,0.349408,4.119763,0.343314,0.846485,0.069813,13.198245
3,10005,AMANCE,48.291854,4.487049,2000,0,0,10140,['AUBE'],['CHAMPAGNE-ARDENNE'],...,11.527883,24.0,2.0,4.238762,0.353230,4.164776,0.347065,0.842121,0.078540,4.751647
4,10006,ARCIS-SUR-AUBE,48.527827,4.141971,2000,0,0,10700,['AUBE'],['CHAMPAGNE-ARDENNE'],...,11.784230,24.0,2.0,4.209328,0.350777,4.135861,0.344655,0.846485,0.074176,8.537810


In [89]:
df.columns

Index(['insee_1', 'nom_commune_1', 'latitude_1', 'longitude_1', 'year_1',
       'applied_1', 'decision_1', 'Code Postal_1', 'Département_1', 'Région_1',
       'Code Département_1', 'Code Région_1', 'latitude_rad_1',
       'longitude_rad_1', 'latitude_2', 'longitude_2', 'year_2', 't2m_z_sum_2',
       't2m_z_mean_2', 'tp_z_sum_2', 'tp_z_mean_2', 'e_z_sum_2', 'e_z_mean_2',
       'pev_z_sum_2', 'pev_z_mean_2', 'stl1_z_sum_2', 'stl1_z_mean_2',
       'stl2_z_sum_2', 'stl2_z_mean_2', 'slt_z_sum_2', 'slt_z_mean_2',
       'swvl1_z_sum_2', 'swvl1_z_mean_2', 'swvl2_z_sum_2', 'swvl2_z_mean_2',
       't2m_sum_2', 't2m_mean_2', 'tp_sum_2', 'tp_mean_2', 'e_sum_2',
       'e_mean_2', 'pev_sum_2', 'pev_mean_2', 'stl1_sum_2', 'stl1_mean_2',
       'stl2_sum_2', 'stl2_mean_2', 'slt_sum_2', 'slt_mean_2', 'swvl1_sum_2',
       'swvl1_mean_2', 'swvl2_sum_2', 'swvl2_mean_2', 'latitude_rad_2',
       'longitude_rad_2', 'distance_km'],
      dtype='object')

In [90]:
df.dtypes

insee_1                object
nom_commune_1          object
latitude_1            float64
longitude_1           float64
year_1                  int64
applied_1               int64
decision_1              int64
Code Postal_1          object
Département_1          object
Région_1               object
Code Département_1     object
Code Région_1           int64
latitude_rad_1        float64
longitude_rad_1       float64
latitude_2            float64
longitude_2           float64
year_2                  int32
t2m_z_sum_2           float64
t2m_z_mean_2          float64
tp_z_sum_2            float64
tp_z_mean_2           float64
e_z_sum_2             float64
e_z_mean_2            float64
pev_z_sum_2           float64
pev_z_mean_2          float64
stl1_z_sum_2          float64
stl1_z_mean_2         float64
stl2_z_sum_2          float64
stl2_z_mean_2         float64
slt_z_sum_2           float64
slt_z_mean_2          float64
swvl1_z_sum_2         float64
swvl1_z_mean_2        float64
swvl2_z_su

In [91]:
df.select_dtypes(include=['object']).describe()

,insee_1,nom_commune_1,Code Postal_1,Département_1,Région_1,Code Département_1
count,868454,868454,868454,868454,868454,868454
unique,35958,33400,5862,96,22,96
top,47032,SAINTE-COLOMBE,51300,['PAS-DE-CALAIS'],['MIDI-PYRENEES'],62
freq,37,339,1104,21034,71376,21034


In [92]:
df.select_dtypes(include=['int32', 'int64']).describe()

,year_1,applied_1,decision_1,Code Région_1,year_2
count,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000
mean,2011.496121,0.445718,0.038123,49.226947,2011.496121
std,6.910162,0.497045,0.191493,25.119029,6.910162
min,2000.000000,0.000000,0.000000,11.000000,2000.000000
25%,2006.000000,0.000000,0.000000,25.000000,2006.000000
50%,2011.000000,0.000000,0.000000,43.000000,2011.000000
75%,2017.000000,1.000000,0.000000,73.000000,2017.000000
max,2023.000000,1.000000,1.000000,94.000000,2023.000000


In [93]:
df.select_dtypes(include=['float32', 'float64']).describe()

,latitude_1,longitude_1,latitude_rad_1,longitude_rad_1,latitude_2,longitude_2,t2m_z_sum_2,t2m_z_mean_2,tp_z_sum_2,tp_z_mean_2,...,stl2_mean_2,slt_sum_2,slt_mean_2,swvl1_sum_2,swvl1_mean_2,swvl2_sum_2,swvl2_mean_2,latitude_rad_2,longitude_rad_2,distance_km
count,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,...,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000,868454.000000
mean,47.018240,2.685467,0.820623,0.046870,47.018081,2.685476,4.617781,0.384815,-0.314113,-0.026176,...,11.651255,28.554300,2.379525,3.882993,0.323583,3.799258,0.316605,0.820620,0.046870,9.042732
std,2.164815,2.596413,0.037783,0.045316,2.166321,2.596766,4.183193,0.348599,3.158859,0.263238,...,1.704982,7.760094,0.646675,0.572538,0.047712,0.588889,0.049074,0.037809,0.045322,3.553215
min,42.285250,-5.085345,0.738017,-0.088756,42.500000,-5.000000,-9.260896,-0.771741,-9.733873,-0.811156,...,1.713064,0.000000,0.000000,-0.000040,-0.000003,-0.000050,-0.000004,0.741765,-0.087266,0.060873
25%,45.239864,0.659644,0.789585,0.011513,45.250000,0.750000,1.546452,0.128871,-2.616450,-0.218038,...,10.737213,24.000000,2.000000,3.713022,0.309419,3.587671,0.298973,0.789761,0.013090,6.460281
50%,47.426567,2.607542,0.827750,0.045510,47.500000,2.500000,5.069161,0.422430,-0.272641,-0.022720,...,11.595448,24.000000,2.000000,3.944875,0.328740,3.855600,0.321300,0.829031,0.043633,9.150935
75%,48.843291,4.821226,0.852476,0.084146,48.750000,4.750000,7.893278,0.657773,1.910088,0.159174,...,12.635692,36.000000,3.000000,4.165593,0.347133,4.115305,0.342942,0.850848,0.082903,11.835353
max,51.063772,8.825756,0.891231,0.154038,51.000000,8.500000,17.793317,1.482776,13.873955,1.156163,...,19.520823,48.000000,4.000000,5.718729,0.476561,5.756094,0.479674,0.890118,0.148353,35.531838


In [94]:
def plot_distance_boxplot(df):
    fig = px.box(df, y='distance_km', title='Boxplot of Distance (km)', labels={'distance_km': 'Distance (km)'})
    fig.show()

# Example usage for the plot:
# plot_distance_boxplot(df)

In [95]:
df.columns

Index(['insee_1', 'nom_commune_1', 'latitude_1', 'longitude_1', 'year_1',
       'applied_1', 'decision_1', 'Code Postal_1', 'Département_1', 'Région_1',
       'Code Département_1', 'Code Région_1', 'latitude_rad_1',
       'longitude_rad_1', 'latitude_2', 'longitude_2', 'year_2', 't2m_z_sum_2',
       't2m_z_mean_2', 'tp_z_sum_2', 'tp_z_mean_2', 'e_z_sum_2', 'e_z_mean_2',
       'pev_z_sum_2', 'pev_z_mean_2', 'stl1_z_sum_2', 'stl1_z_mean_2',
       'stl2_z_sum_2', 'stl2_z_mean_2', 'slt_z_sum_2', 'slt_z_mean_2',
       'swvl1_z_sum_2', 'swvl1_z_mean_2', 'swvl2_z_sum_2', 'swvl2_z_mean_2',
       't2m_sum_2', 't2m_mean_2', 'tp_sum_2', 'tp_mean_2', 'e_sum_2',
       'e_mean_2', 'pev_sum_2', 'pev_mean_2', 'stl1_sum_2', 'stl1_mean_2',
       'stl2_sum_2', 'stl2_mean_2', 'slt_sum_2', 'slt_mean_2', 'swvl1_sum_2',
       'swvl1_mean_2', 'swvl2_sum_2', 'swvl2_mean_2', 'latitude_rad_2',
       'longitude_rad_2', 'distance_km'],
      dtype='object')

In [96]:
z_columns = [col for col in df.columns if '_z_' in col]
z_columns

['t2m_z_sum_2',
 't2m_z_mean_2',
 'tp_z_sum_2',
 'tp_z_mean_2',
 'e_z_sum_2',
 'e_z_mean_2',
 'pev_z_sum_2',
 'pev_z_mean_2',
 'stl1_z_sum_2',
 'stl1_z_mean_2',
 'stl2_z_sum_2',
 'stl2_z_mean_2',
 'slt_z_sum_2',
 'slt_z_mean_2',
 'swvl1_z_sum_2',
 'swvl1_z_mean_2',
 'swvl2_z_sum_2',
 'swvl2_z_mean_2']

In [97]:
df['decision_1'].describe()

count    868454.000000
mean          0.038123
std           0.191493
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: decision_1, dtype: float64

In [103]:
df['decision_1'].value_counts()

decision_1
0    353978
1     33108
Name: count, dtype: int64

## Models

### logistic Regression

#### Basic Logistic Regression

In [102]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Custom transformer to extract columns containing '_z_'
class ZColumnExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        # Selecting only columns that contain '_z_' in their name
        z_columns = [col for col in X.columns if '_z_' in col]
        return X[z_columns].copy()

# Custom transformer for latitude and longitude transformations in radians
class LatLonTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        X = X.copy()
        if 'latitude_rad_1' in X.columns and 'longitude_rad_1' in X.columns:
            lat_rad = X['latitude_rad_1']
            lon_rad = X['longitude_rad_1']
            # Apply sine and cosine transformations
            X['lat_sin'] = np.sin(lat_rad)
            X['lat_cos'] = np.cos(lat_rad)
            X['lon_sin'] = np.sin(lon_rad)
            X['lon_cos'] = np.cos(lon_rad)
            # Dropping original latitude and longitude columns
            X = X.drop(columns=['latitude_rad_1', 'longitude_rad_1'])
        return X

# Assuming df is already defined in your environment
df_sorted = df.sort_values(by='year_1')  # Sorting by year to maintain temporal consistency

# Define X and y after sorting
X_sorted = df_sorted.drop(columns=['decision_1'])
y_sorted = df_sorted['decision_1']

# Define a cut-off year for training and testing
cutoff_year = df_sorted['year_1'].quantile(0.8)  # 80% of the data for training, 20% for testing

# Split data into training and testing based on the cutoff year
X_train = X_sorted.loc[df_sorted['year_1'] <= cutoff_year]
y_train = y_sorted.loc[df_sorted['year_1'] <= cutoff_year]
X_test = X_sorted.loc[df_sorted['year_1'] > cutoff_year]
y_test = y_sorted.loc[df_sorted['year_1'] > cutoff_year]

# Creating the column transformer to handle different preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('lat_lon_transformer', LatLonTransformer(), ['latitude_rad_1', 'longitude_rad_1']),  # Transform latitude and longitude (in radians)
        ('z_column_extractor', ZColumnExtractor(), X_train.columns),  # Extract columns with '_z_'
        ('scaler', StandardScaler(), [col for col in X_train.columns if '_z_' in col]),  # Scaling the selected columns
          # Pass through the 'applied' column as is
    ],
    remainder='drop'  # Drop other columns not specified in transformers
)

# Creating the pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced'))  # Handle imbalance with class_weight='balanced'
])

# Fit the pipeline to X_train (features) and transform X_train
pipeline.fit(X_train, y_train)

# Transform the test set and make predictions
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]  # Probabilities for the positive class

# Calculate and print evaluation metrics
# Classification Report (Precision, Recall, F1 Score)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

# Precision-Recall Curve and Average Precision Score (PR-AUC)
average_precision = average_precision_score(y_test, y_prob)
print(f"Average Precision (PR-AUC): {average_precision:.2f}")


Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.43      0.59     53468
           1       0.24      0.93      0.39     10582

    accuracy                           0.51     64050
   macro avg       0.61      0.68      0.49     64050
weighted avg       0.85      0.51      0.56     64050

Confusion Matrix:
[[22760 30708]
 [  721  9861]]
ROC-AUC Score: 0.77
Average Precision (PR-AUC): 0.36


#### Logistic Regression With Smote 

In [113]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.base import BaseEstimator, TransformerMixin
import warnings

warnings.filterwarnings('ignore')  # Ignore warnings for a cleaner output

##################################
# Step 1: Data Filtering and Preparation
##################################

# Filter columns that match '_z_' for weather data and add latitude/longitude features
weather_features = [col for col in df.columns if '_z_' in col]
spatial_features = ['latitude_rad_1', 'longitude_rad_1']
selected_features = weather_features + spatial_features

# Define X and y
X = df[selected_features]
y = df['decision_1']  # Target column is 'decision_1', which contains 0 or 1 values
X['year_1'] = df['year_1']  # Add 'year_1' information to preserve temporality

##################################
# Step 2: Train-Test Split (Chronological)
##################################

# Sort data chronologically by 'year_1' to maintain temporal order
df = df.sort_values(by=['year_1'])

# Split data into training (80%) and testing (20%) while maintaining chronological order
train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

##################################
# Step 3: Define Preprocessing Pipeline
##################################

# Define a custom transformer to apply sine and cosine transformations to latitude and longitude
class SineCosineTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[f'{col}_sin'] = np.sin(X[col])
            X[f'{col}_cos'] = np.cos(X[col])
        return X.drop(columns=self.columns)  # Drop original columns after transformation

# Define the columns for sine and cosine transformation
trigonometric_columns = ['latitude_rad_1', 'longitude_rad_1']

# Create the sine-cosine transformer
sine_cosine_transformer = SineCosineTransformer(columns=trigonometric_columns)

# Define preprocessing for numerical features (imputation and scaling)
numerical_features = weather_features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Impute missing values with mean
    ('scaler', StandardScaler())  # Standard scaling
])

# Combine preprocessing for numerical features and trigonometric transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('trig', sine_cosine_transformer, trigonometric_columns)
    ]
)

##################################
# Step 4: Apply Preprocessing to Training and Test Data
##################################

# Preprocess the training and test data
X_train_preprocessed = preprocessor.fit_transform(X_train.drop(columns=['year_1']))
X_test_preprocessed = preprocessor.transform(X_test.drop(columns=['year_1']))

##################################
# Step 5: Apply SMOTE Within Each Year
##################################

# Add the 'year_1' information back to X_train_preprocessed for grouping
X_train_preprocessed = pd.DataFrame(X_train_preprocessed)
X_train_preprocessed['year_1'] = X_train['year_1'].values
y_train_reset = y_train.reset_index(drop=True)

# Create lists to store resampled data for each year_1
X_train_resampled_list = []
y_train_resampled_list = []

# Apply SMOTE separately for each year_1 to avoid data leakage
for year_1 in X_train_preprocessed['year_1'].unique():
    # Extract data for the current year_1
    X_year = X_train_preprocessed[X_train_preprocessed['year_1'] == year_1].drop(columns=['year_1'])
    y_year = y_train_reset[X_train_preprocessed['year_1'] == year_1]
    
    # Apply SMOTE only if the minority class has enough samples
    if len(y_year[y_year == 1]) > 1:
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X_year, y_year)
    else:
        # If not enough minority samples, keep original data
        X_resampled, y_resampled = X_year, y_year
    
    # Append the resampled data for this year_1
    X_train_resampled_list.append(X_resampled)
    y_train_resampled_list.append(y_resampled)

# Concatenate the resampled data for all years to form the final training set
X_train_resampled = pd.concat(X_train_resampled_list, axis=0)
y_train_resampled = pd.concat(y_train_resampled_list, axis=0)

##################################
# Step 6: Train the Model Using Logistic Regression
##################################

# Initialize Logistic Regression model
classifier = LogisticRegression()

# Train the model on the resampled training data
classifier.fit(X_train_resampled, y_train_resampled)

# Make predictions on the original test set
y_pred_class = classifier.predict(X_test_preprocessed)

##################################
# Step 7: Evaluate the Model
##################################

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred_class)
conf_matrix = confusion_matrix(y_test, y_pred_class)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", classification_report(y_test, y_pred_class))

"""
Summary:
1. Data was filtered to include relevant weather and spatial features.
2. Data was split chronologically to preserve temporal integrity.
3. Preprocessing included scaling numerical features and applying sine/cosine transformations to spatial features.
4. SMOTE was applied within each year_1 to balance the dataset while maintaining temporal order.
5. Logistic Regression was used to train the model on the balanced training set.
6. The model was evaluated on the original test set, maintaining the integrity of the temporal split.
"""


Accuracy: 0.6075460487225193
Confusion Matrix:
 [[40215 24198]
 [ 6185  6820]]
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.62      0.73     64413
           1       0.22      0.52      0.31     13005

    accuracy                           0.61     77418
   macro avg       0.54      0.57      0.52     77418
weighted avg       0.76      0.61      0.66     77418



'\nSummary:\n1. Data was filtered to include relevant weather and spatial features.\n2. Data was split chronologically to preserve temporal integrity.\n3. Preprocessing included scaling numerical features and applying sine/cosine transformations to spatial features.\n4. SMOTE was applied within each year_1 to balance the dataset while maintaining temporal order.\n5. Logistic Regression was used to train the model on the balanced training set.\n6. The model was evaluated on the original test set, maintaining the integrity of the temporal split.\n'

#### Logistic Regression with Smote and Threshold tunning

In [119]:
# This script aims to classify weather data with an imbalanced target variable ('decision_1') using Logistic Regression.
# Key steps include data preprocessing, applying SMOTE (Synthetic Minority Over-sampling Technique) to balance the classes,
# preserving temporal data integrity by processing data year-by-year, and fine-tuning model evaluation thresholds for better performance.
# The goal is to maximize the predictive power of the model while preventing data leakage and respecting temporal relationships.

# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.base import BaseEstimator, TransformerMixin
import warnings

warnings.filterwarnings('ignore')  # Ignore warnings for a cleaner output

##################################
# Step 1: Data Filtering and Preparation
##################################

# Filter columns that match '_z_' for weather data and add latitude/longitude features
weather_features = [col for col in df.columns if '_z_' in col]
spatial_features = ['latitude_rad_1', 'longitude_rad_1']
selected_features = weather_features + spatial_features

# Define X and y
X = df[selected_features]
y = df['decision_1']  # Target column is 'decision_1', which contains 0 or 1 values
X['year_1'] = df['year_1']  # Add 'year_1' information to preserve temporality

##################################
# Step 2: Train-Test Split (Chronological)
##################################

# Sort data chronologically by 'year_1' to maintain temporal order
df = df.sort_values(by=['year_1'])

# Split data into training (80%) and testing (20%) while maintaining chronological order
train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

##################################
# Step 3: Define Preprocessing Pipeline
##################################

# Define a custom transformer to apply sine and cosine transformations to latitude and longitude
class SineCosineTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[f'{col}_sin'] = np.sin(X[col])
            X[f'{col}_cos'] = np.cos(X[col])
        return X.drop(columns=self.columns)  # Drop original columns after transformation

# Define the columns for sine and cosine transformation
trigonometric_columns = ['latitude_rad_1', 'longitude_rad_1']

# Create the sine-cosine transformer
sine_cosine_transformer = SineCosineTransformer(columns=trigonometric_columns)

# Define preprocessing for numerical features (imputation and scaling)
numerical_features = weather_features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Impute missing values with mean
    ('scaler', StandardScaler())  # Standard scaling
])

# Combine preprocessing for numerical features and trigonometric transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('trig', sine_cosine_transformer, trigonometric_columns)
    ]
)

##################################
# Step 4: Apply Preprocessing to Training and Test Data
##################################

# Preprocess the training and test data
X_train_preprocessed = preprocessor.fit_transform(X_train.drop(columns=['year_1']))
X_test_preprocessed = preprocessor.transform(X_test.drop(columns=['year_1']))

##################################
# Step 5: Apply SMOTE Within Each Year
##################################

# Add the 'year_1' information back to X_train_preprocessed for grouping
X_train_preprocessed = pd.DataFrame(X_train_preprocessed)
X_train_preprocessed['year_1'] = X_train['year_1'].values
y_train_reset = y_train.reset_index(drop=True)

# Create lists to store resampled data for each year_1
X_train_resampled_list = []
y_train_resampled_list = []

# Apply SMOTE separately for each year_1 to avoid data leakage
for year_1 in X_train_preprocessed['year_1'].unique():
    # Extract data for the current year_1
    X_year = X_train_preprocessed[X_train_preprocessed['year_1'] == year_1].drop(columns=['year_1'])
    y_year = y_train_reset[X_train_preprocessed['year_1'] == year_1]
    
    # Apply SMOTE only if the minority class has enough samples
    if len(y_year[y_year == 1]) > 1:
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X_year, y_year)
    else:
        # If not enough minority samples, keep original data
        X_resampled, y_resampled = X_year, y_year
    
    # Append the resampled data for this year_1
    X_train_resampled_list.append(X_resampled)
    y_train_resampled_list.append(y_resampled)

# Concatenate the resampled data for all years to form the final training set
X_train_resampled = pd.concat(X_train_resampled_list, axis=0)
y_train_resampled = pd.concat(y_train_resampled_list, axis=0)

##################################
# Step 6: Train the Model Using Logistic Regression


##################################

# Initialize Logistic Regression model with class weight balancing
classifier = LogisticRegression(class_weight='balanced')


# Train the model on the resampled training data
classifier.fit(X_train_resampled, y_train_resampled)

# Make predictions on the original test set
y_pred_class = classifier.predict(X_test_preprocessed)

##################################
# Step 7: Evaluate the Model with Threshold Tuning
##################################

# Predict probabilities instead of direct classes for threshold tuning
y_pred_prob = classifier.predict_proba(X_test_preprocessed)[:, 1]

# Define a function to evaluate the model with different thresholds
def evaluate_with_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    accuracy = accuracy_score(y_true, y_pred)
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(f"Threshold: {threshold}")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:", conf_matrix)
    print("Classification Report:", classification_report(y_true, y_pred))

# Test different thresholds to find the best one
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
for threshold in thresholds:
    evaluate_with_threshold(y_test, y_pred_prob, threshold)
accuracy = accuracy_score(y_test, y_pred_class)
conf_matrix = confusion_matrix(y_test, y_pred_class)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", classification_report(y_test, y_pred_class))

"""
Summary:
1. Data was filtered to include relevant weather and spatial features.
2. Data was split chronologically to preserve temporal integrity.
3. Preprocessing included scaling numerical features and applying sine/cosine transformations to spatial features.
4. SMOTE was applied within each year_1 to balance the dataset while maintaining temporal order.
5. Logistic Regression was used to train the model on the balanced training set.
6. The model was evaluated on the original test set, maintaining the integrity of the temporal split.
"""


Threshold: 0.3
Accuracy: 0.4055258467023173
Confusion Matrix: [[21316 43099]
 [ 2924 10079]]
Classification Report:               precision    recall  f1-score   support

           0       0.88      0.33      0.48     64415
           1       0.19      0.78      0.30     13003

    accuracy                           0.41     77418
   macro avg       0.53      0.55      0.39     77418
weighted avg       0.76      0.41      0.45     77418

Threshold: 0.4
Accuracy: 0.5098814229249012
Confusion Matrix: [[30974 33441]
 [ 4503  8500]]
Classification Report:               precision    recall  f1-score   support

           0       0.87      0.48      0.62     64415
           1       0.20      0.65      0.31     13003

    accuracy                           0.51     77418
   macro avg       0.54      0.57      0.46     77418
weighted avg       0.76      0.51      0.57     77418

Threshold: 0.5
Accuracy: 0.6073781291172595
Confusion Matrix: [[40206 24209]
 [ 6187  6816]]
Classification Report

'\nSummary:\n1. Data was filtered to include relevant weather and spatial features.\n2. Data was split chronologically to preserve temporal integrity.\n3. Preprocessing included scaling numerical features and applying sine/cosine transformations to spatial features.\n4. SMOTE was applied within each year_1 to balance the dataset while maintaining temporal order.\n5. Logistic Regression was used to train the model on the balanced training set.\n6. The model was evaluated on the original test set, maintaining the integrity of the temporal split.\n'